# Кандидатогенерация для поиска услуг Авито

**Задача:** для каждого поискового запроса вернуть до 50 объявлений из корпуса (189 212 шт.) так,
чтобы в них попали объявления, которые пользователь в итоге выбрал. Метрика — Recall@50, усреднённый по запросам.

**Результат на платформе:** Recall@50 = **0.876**. Локальная валидация: 0.942 (первый этап без ранкера 0.915, потолок кандидатов 0.982).

Разрыв объясняется тем, что валидационные запросы из train проще бенчмарковых: в среднем 2.5 слова против 3.2, фильтры в 64% запросов против 37%, поиск по региону в 14% против 17%. Следующий шаг — стратифицировать валидацию и обучение ранкера под распределение бенчмарка.

**Схема решения**

1. **Этап 1 — дешёвый отбор ~300 кандидатов** из всего корпуса по сумме четырёх сигналов:
   * текст: BM25 по заголовку, параметрам и описанию (Snowball-стемминг) + BM25 по «расширению документа»
     (запросы из train, по которым выбирали это объявление) + символьные n-граммы заголовка (опечатки);
   * локация: P(локация объявления | локация поиска), оценённая по train, со сглаживанием по расстоянию;
   * подкатегория: классификатор P(microcat | запрос, фильтры поиска), обученный на train;
2. **Этап 2 — LightGBM (lambdarank)** переупорядочивает кандидатов по ~40 признакам, берём top-50.

Код этапов вынесен в `src/`, здесь — последовательный прогон с пояснениями, валидацией и разбором ошибок.
Seed'ы зафиксированы. В LightGBM стоят `deterministic=True` и `force_col_wise=True`: без последнего параметра LightGBM выбирает режим построения по замеру скорости, и повторные прогоны расходились примерно на 2% объявлений в ответах.

In [1]:
import sys, os, time, hashlib
import numpy as np, pandas as pd, lightgbm as lgb
sys.path.insert(0, 'src')
from common import make_folds, recall_at_k, GROUP
from prep import item_fields
from pipeline import ItemIndex, TrainStats, generate, label, FEATS

DATA = 'dataset'          # папка с train.parquet, benchmark_items.parquet, benchmark_queries.parquet
OUT = 'answer.csv'
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 70)
t0 = time.time(); log = lambda *a: print(f'[{time.time() - t0:6.0f}s]', *a, flush=True)

## 1. Данные и что из них следует

Текстовые признаки строятся только по корпусу бенчмарка. Из train нужны запросы, а у выбранных объявлений —
id, локация, подкатегория и координаты. Поэтому train читаем без `item_description_raw`: это экономит память.

In [2]:
items = pd.read_parquet(f'{DATA}/benchmark_items.parquet')
train = pd.read_parquet(f'{DATA}/train.parquet', columns=[
    'search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category',
    'item_id', 'item_location_id', 'item_microcat_id', 'item_latitude', 'item_longitude'])
bench = pd.read_parquet(f'{DATA}/benchmark_queries.parquet')
print('train', train.shape, '| items', items.shape, '| bench', bench.shape)

corpus_ids = set(items.item_id)
print('объявлений корпуса, встречавшихся в train:      ', round(len(corpus_ids & set(train.item_id)) / len(corpus_ids), 3))
print('запросов бенчмарка, чей текст встречался в train:', round(bench.search_query.isin(set(train.search_query)).mean(), 3))
print('пар train, где локация объявления = локации поиска:', round((train.search_location_id == train.item_location_id).mean(), 3))
print('локаций поиска бенчмарка без объявлений в этой же локации:',
      round((~bench.search_location_id.isin(items.item_location_id)).mean(), 3))

train (497673, 10) | items (189212, 14) | bench (2452, 6)


объявлений корпуса, встречавшихся в train:       0.096


запросов бенчмарка, чей текст встречался в train: 0.37
пар train, где локация объявления = локации поиска: 0.831
локаций поиска бенчмарка без объявлений в этой же локации: 0.174


In [3]:
# Локация поиска бывает городом (выбирают почти всегда там же) или регионом/страной (выбирают в городах внутри)
m = train.groupby('search_location_id').item_location_id.agg(lambda s: s.value_counts(normalize=True).head(3).round(3).to_dict())
for l in bench.search_location_id.value_counts().index[:6]:
    print(l, '| запросов в бенчмарке:', (bench.search_location_id == l).sum(),
          '| объявлений в этой локации:', (items.item_location_id == l).sum(), '| top-3 P(item_loc):', m.get(l))

637640 | запросов в бенчмарке: 329 | объявлений в этой локации: 19543 | top-3 P(item_loc): {637640: 0.921, 637641: 0.013, 639740: 0.005}
653240 | запросов в бенчмарке: 214 | объявлений в этой локации: 12145 | top-3 P(item_loc): {653240: 0.944, 806660: 0.007, 636840: 0.007}
107620 | запросов в бенчмарке: 119 | объявлений в этой локации: 0 | top-3 P(item_loc): {637640: 0.612, 638790: 0.028, 637770: 0.024}
621540 | запросов в бенчмарке: 94 | объявлений в этой локации: 0 | top-3 P(item_loc): {637640: 0.273, 653240: 0.107, 633540: 0.032}
650400 | запросов в бенчмарке: 75 | объявлений в этой локации: 3659 | top-3 P(item_loc): {650400: 0.979, 650350: 0.002, 637640: 0.002}
633540 | запросов в бенчмарке: 66 | объявлений в этой локации: 4494 | top-3 P(item_loc): {633540: 0.973, 634450: 0.002, 645780: 0.002}


**Выводы, определившие решение**

* **Локация — самый сильный сигнал.** 83% выборов сделаны в той же локации, где искали. Но есть
  «региональные» локации поиска (например, область или вся Россия), где в самой локации объявлений нет.
  Поэтому нужен не жёсткий фильтр, а распределение P(item_loc | search_loc), выученное по train.
* **Корпус почти не пересекается с train** (≈10% объявлений), поэтому нельзя строить решение на «памяти»
  запрос → объявление. Обобщаются тексты и подкатегории: в train ~500 тыс. пар «запрос → подкатегория
  выбранного объявления», и 99% объявлений корпуса лежат в этих подкатегориях.
* **Только 37% текстов запросов бенчмарка встречались в train** — валидация должна это учитывать.
* Запросы короткие (медиана 3 слова), а объявления длинные. Нужна морфология («ремонт стиральных машин»
  ~ «ремонт стиральной машины») и устойчивость к опечаткам.

## 2. Валидация

Разметки для бенчмарка нет, поэтому валидацию строим из train, стараясь повторить условия теста:

* единица оценки — «поиск» = (текст, локация, фильтры, категория); релевантные — выбранные объявления;
* берём только поиски, чьи выбранные объявления есть в корпусе бенчмарка: поиск идёт по тому же корпусу, что и в тесте;
* для 45% отложенных поисков из обучающей части удаляются **все** строки с тем же текстом запроса.
  Без этого «знакомых» запросов в валидации было бы 87%, а в бенчмарке их 37%, и оценка вышла бы завышенной;
* для 70% отложенных объявлений удаляется их история в train. Иначе признаки «объявление уже выбирали»
  переоценивались бы: у валидации таких объявлений 32%, а в корпусе в среднем 10%.

Отложенных поисков 15 000, это 3 фолда по 5 000. Для каждого фолда статистики первого этапа считаются
по train без этого фолда. Так ранкер учится на 15 тыс. запросов, а статистики почти не теряют данных.
Честная оценка: ранкер учится на фолдах 1–2, метрика считается на фолде 0.

In [4]:
idx = ItemIndex(items, item_fields(items, cache='cache/items_tok.pkl')); log('индекс корпуса готов')
GEN = dict(K=300, A=4, C=2, G=0.3, H=0.15)   # параметры этапа 1, подобраны на валидации (см. абляцию ниже)

folds = make_folds(train, corpus_ids, n_folds=3, per_fold=5000)
C, st0 = [], None
for k, (fit, q, rel) in enumerate(folds):
    st = TrainStats(fit, idx)
    d = generate(q, st, **GEN); d['y'] = label(d, rel, idx)
    d[FEATS] = d[FEATS].astype(np.float32); d['qi'] += k * 10 ** 6
    C.append((d, rel)); log(f'фолд {k}: train_fit {len(fit)} строк, кандидатов {len(d)}')
    if k == 0:
        st0, q0 = st, q          # сохраняем для абляции и разбора ошибок
del folds, st

[    55s] индекс корпуса готов


[   343s] фолд 0: train_fit 284298 строк, кандидатов 1554733


[   623s] фолд 1: train_fit 283762 строк, кандидатов 1554757


[   896s] фолд 2: train_fit 292712 строк, кандидатов 1555406


## 3. Этап 1: каналы и их вклад (абляция)

Скор кандидата на первом этапе:

$$\text{base} = \underbrace{\text{BM25}_{title} + \text{BM25}_{params} + \text{BM25}_{desc} + \text{BM25}_{exp} + 1.5\cdot\text{char}}_{\text{text}}
\;+\; A\cdot\log\big(P(loc) + G\,e^{-d/30\text{км}} + \varepsilon\big) \;+\; C\cdot\log\big(P(microcat) + \varepsilon\big)$$

* BM25 реализован разреженными матрицами (scipy): для батча из 100 запросов скор по всем 189 тыс. объявлений считается одним умножением;
* $d$ — расстояние от «центра» локации поиска (медиана координат объявлений, выбранных в ней в train) до объявления.
  Сглаживание нужно для соседних городов, пара которых ни разу не встречалась в train;
* классификатор подкатегории — TF-IDF (униграммы и биграммы стемов запроса + токены фильтров) и логистическая регрессия (SGD).

Ниже Recall@50 первого этапа (без ранкера) на 1 500 запросах фолда 0 при последовательном добавлении каналов.

In [5]:
qa = q0.head(1500); ra = C[0][1][:1500]
def stage1_recall(cfg):
    d = generate(qa, st0, **cfg)
    d = d.sort_values(['qi', 'base', 'row'], ascending=[True, False, True], kind='stable').groupby('qi').head(50)
    hit = d.assign(h=label(d, ra, idx)).groupby('qi').h.sum().reindex(range(len(ra)), fill_value=0)
    return float(np.mean(hit.values / np.array([len(r) for r in ra])))
abl = [('только текст (BM25 + char)',              dict(K=300, A=0, C=0, G=0,   H=0.15)),
       ('+ локация P(item_loc | search_loc)',      dict(K=300, A=4, C=0, G=0,   H=0.15)),
       ('+ подкатегория P(microcat | запрос)',     dict(K=300, A=4, C=2, G=0,   H=0.15)),
       ('+ гео-сглаживание (итоговый этап 1)',     dict(K=300, A=4, C=2, G=0.3, H=0.15))]
res = [(name, round(stage1_recall(cfg), 4)) for name, cfg in abl]
pd.DataFrame(res, columns=['этап 1', 'Recall@50'])

,этап 1,Recall@50
0,только текст (BM25 + char),0.2788
1,+ локация P(item_loc | search_loc),0.8717
2,+ подкатегория P(microcat | запрос),0.9000
3,+ гео-сглаживание (итоговый этап 1),0.8993


## 4. Этап 2: ранкер

LightGBM с целевой функцией `lambdarank` на кандидатах этапа 1. Признаки:

* скоры каналов: BM25 по полям, char-n-граммы, «расширение документа», P(loc), P(microcat), расстояние в км, итоговый `base` и его ранг;
* покрытие: доля слов запроса в заголовке, доля слов фильтров поиска в параметрах объявления, выполнение фильтра «рейтинг 4+»;
* объявление: рейтинг, число отзывов, наличие цены, длина описания, популярность в train;
* **относительные признаки внутри запроса:** разница с лучшим кандидатом и ранг по скору, P(microcat), отзывам.
  Они дали +0.6 п.п.: ранжированию важен не абсолютный скор, а отрыв от лидера;
* запрос: длина, встречался ли текст в train, число кандидатов в той же локации.

Обучаем только на запросах, где среди кандидатов есть хотя бы одно релевантное объявление.

In [6]:
LGB = dict(objective='lambdarank', learning_rate=0.05, num_leaves=63, min_data_in_leaf=50, feature_fraction=0.8,
           bagging_fraction=0.8, bagging_freq=1, lambdarank_truncation_level=60,
           seed=0, deterministic=True, force_col_wise=True, num_threads=2, verbose=-1)
N_ROUNDS = 300

def fit_ranker(d):
    d = d[d.groupby('qi').y.transform('max') > 0].sort_values(['qi', 'row'], kind='stable')
    return lgb.train(LGB, lgb.Dataset(d[FEATS], d.y, group=d.groupby('qi', sort=True).size().values), N_ROUNDS)

def recall50(d, rel):
    d = d.sort_values(['qi', 'p', 'row'], ascending=[True, False, True], kind='stable').groupby('qi').head(50)
    hit = d.assign(h=label(d, rel, idx)).groupby('qi').h.sum().reindex(range(len(rel)), fill_value=0)
    return float(np.mean(hit.values / np.array([len(r) for r in rel])))

m_eval = fit_ranker(pd.concat([C[1][0], C[2][0]], ignore_index=True))
dv, rv = C[0]; dv['p'] = m_eval.predict(dv[FEATS])
ceil = dv.groupby('qi').y.sum().reindex(range(len(rv)), fill_value=0).values / np.array([len(r) for r in rv])
print('ВАЛИДАЦИЯ (фолд 0, 5000 запросов)')
print('  Recall@50 этап 1 (base):  ', round(recall50(dv.assign(p=dv['base']), rv), 4))
print('  Recall@50 этап 1 + ранкер:', round(recall50(dv, rv), 4))
print('  потолок (доля релевантных среди ~310 кандидатов):', round(float(ceil.mean()), 4))
imp = pd.Series(m_eval.feature_importance('gain'), FEATS).sort_values(ascending=False)
(imp / imp.sum()).round(3).head(15)

ВАЛИДАЦИЯ (фолд 0, 5000 запросов)


  Recall@50 этап 1 (base):   0.9146


  Recall@50 этап 1 + ранкер: 0.9424
  потолок (доля релевантных среди ~310 кандидатов): 0.9819


base_rank       0.520
base_dmax       0.107
base            0.054
loc_p_dmax      0.043
dist_km         0.021
s_desc          0.018
text            0.017
reviews_rank    0.015
s_params        0.014
s_desc_dmax     0.013
loc_p           0.012
s_char          0.011
desc_len        0.011
mc_p            0.011
s_desc_rank     0.011
dtype: float64

## 5. Разбор ошибок

Смотрим на релевантные объявления фолда 0, которые не попали в top-50: не нашлись среди кандидатов
(проблема этапа 1) или нашлись, но ранкер поставил их ниже 50-го места.

In [7]:
dv['prank'] = dv.groupby('qi').p.rank(ascending=False, method='first')
id2row = idx.id2row
e = pd.DataFrame([(qi, id2row[i]) for qi, r in enumerate(rv) for i in r], columns=['qi', 'row'])
e = e.merge(dv[['qi', 'row', 'prank', 'loc_p', 'mc_p', 'dist_km', 'text_rank']], on=['qi', 'row'], how='left')
e['query'] = q0.search_query.values[e.qi]; e['s_loc'] = q0.search_location_id.values[e.qi]
e['title'] = items.item_title_raw.values[e.row]; e['i_loc'] = items.item_location_id.values[e.row]
e['type'] = np.where(e.prank.isna(), 'нет в кандидатах', np.where(e.prank > 50, 'кандидат, но ранг > 50', 'найдено'))
e['seen'] = np.where(q0.search_query.isin(st0.seen_q).values[e.qi], 'текст знаком', 'новый текст')
e['other_city'] = np.where(e.s_loc == e.i_loc, 'та же локация', 'другая локация')
print(e.type.value_counts(normalize=True).round(3).to_string(), '\n')
for col in ['seen', 'other_city']:
    print((e.type != 'найдено').groupby(e[col]).mean().round(3).rename('доля промахов').to_string(), '\n')
# массовые запросы: сколько кандидатов в той же локации с сильным текстовым совпадением
x = e[e.type != 'найдено']
print('самые частые запросы среди промахов:'); print(x['query'].value_counts().head(8).to_string())

type
найдено                   0.939
кандидат, но ранг > 50    0.044
нет в кандидатах          0.017 

seen
новый текст     0.065
текст знаком    0.051 

other_city
другая локация    0.166
та же локация     0.039 

самые частые запросы среди промахов:
query
наращивание ресниц         14
массаж                      9
фотограф                    8
инструктор по вождению      7
установка кондиционеров     4
окрашивание волос           4
контейнер 8м3               4
мастер класс                3


In [8]:
x.sample(20, random_state=0)[['query', 's_loc', 'title', 'i_loc', 'type', 'prank', 'loc_p', 'mc_p', 'dist_km']]

,query,s_loc,title,i_loc,type,prank,loc_p,mc_p,dist_km
3408,ремонт одежды,653240,"Ателье:ремонт одежды,срок 1-2 дня.Без выходных",653240,"кандидат, но ранг > 50",51.0,0.944327,0.912913,9.495557
4354,грузоперевозки межгород,653240,Грузоперевозки газель,653240,"кандидат, но ранг > 50",76.0,0.944327,0.632039,2.009140
2897,вписание доп оборудования авто,641490,Внесение изменений в конструкцию транспорта,641780,нет в кандидатах,NaN,NaN,NaN,NaN
4412,снять нарощенные ресницы бесплатно,637640,Модель брови ресницы,637640,"кандидат, но ранг > 50",104.0,0.926483,0.933813,7.243598
4833,липолитики,904755,Красота и здоровье,652000,нет в кандидатах,NaN,NaN,NaN,NaN
883,заправка сплит систем,624840,"Монтаж, установка кондиционера и сплит систем",624840,"кандидат, но ранг > 50",57.0,0.968940,0.029664,3.130302
258,инструктор по вождению,653240,Автоинструктор на МКПП,653240,"кандидат, но ранг > 50",71.0,0.944327,0.908085,9.982345
4445,сделать временную прописку,653240,Миграционные услуги,653240,нет в кандидатах,NaN,NaN,NaN,NaN
759,брови,653240,"Перманентный макияж, татуаж, волоски",653240,"кандидат, но ранг > 50",113.0,0.944327,0.161717,12.204399
4524,изготовление автомобильных ключей с чипом,637640,"Автоэлектрик, Установка сигнализации",639300,нет в кандидатах,NaN,NaN,NaN,NaN


**Какие ошибки нашлись и что с ними сделано**

| Тип ошибки | Пример | Что сделано |
|---|---|---|
| Текстовый поиск без учёта географии: в top-50 попадают объявления из других городов | почти все запросы | априорная вероятность локации P(item_loc \| search_loc): recall этапа 1 вырос с ~0.30 до ~0.85 |
| Поиск по региону или стране, объявления лежат в городах внутри | «ресницы», поиск по Ленобласти | та же P(item_loc \| search_loc), выученная по train, а не фильтр «локация = локация» |
| Соседний город, пары локаций нет в train | «спил деревьев», соседние районы | гео-сглаживание по расстоянию до центра локации + признак `dist_km` |
| Слова запроса не совпадают со словами объявления | «уроки вождения» → «Автоинструктор», «газовщик сварщик» | классификатор подкатегории по запросу (обучен на 500 тыс. пар train) + «расширение документа» запросами из train |
| Опечатки и транслитерация | «бобкет» → «Бобкат», «кароеды» → «Короед» | символьные n-граммы заголовка (канал и признак `s_char`) |
| Много равноценных объявлений в одном городе | «маникюр», «массаж», «сборка мебели» | ранкер с относительными признаками и сигналами качества (отзывы, рейтинг). Это основной остаток ошибок: выбор пользователя среди сотен одинаковых объявлений почти не предсказывается по тексту |
| Семантические перефразы без общих слов | «установка root прав» → «Разблокировка смартфона», «hummer на прокат» → «Прокачу на гелике» | не решено. Нужны нейросетевые эмбеддинги (например, дообученный e5/bge на парах train). Кандидат на следующий шаг |

## 6. Финальная модель и ответ для бенчмарка

Ранкер переобучается на всех трёх фолдах (15 тыс. запросов). Статистики первого этапа для бенчмарка
считаются по **всему** train.

In [9]:
model = fit_ranker(pd.concat([c[0] for c in C], ignore_index=True)); log('финальный ранкер обучен')
del C, dv, m_eval, st0

st = TrainStats(train, idx); log('статистики по всему train')
d = generate(bench, st, **GEN); log('кандидаты бенчмарка', len(d))
d['p'] = model.predict(d[FEATS])
d = d.sort_values(['qi', 'p', 'row'], ascending=[True, False, True], kind='stable').groupby('qi').head(50)
ans = d.groupby('qi')['row'].agg(lambda r: ' '.join(idx.ids[r.values]))
out = pd.DataFrame({'query_id': bench['query_id'].values, 'answer': [ans.get(i, '') for i in range(len(bench))]})
out.to_csv(OUT, index=False)
log('сохранено', OUT)

[  1607s] финальный ранкер обучен


[  1768s] статистики по всему train


[  1839s] кандидаты бенчмарка 764120


[  1850s] сохранено answer.csv


In [10]:
# Проверка формата по требованиям платформы
import re
a = pd.read_csv(OUT, dtype=str, keep_default_na=False)
assert list(a.columns) == ['query_id', 'answer']
assert a.query_id.is_unique and set(a.query_id) == set(bench.query_id)
for s in a.answer:
    x = s.split(' ')
    assert len(x) <= 50 and len(x) == len(set(x))
    assert all(re.fullmatch(r'[0-9a-f]{16}', i) and i in corpus_ids for i in x)
print('формат OK:', len(a), 'строк; md5 =', hashlib.md5(open(OUT, 'rb').read()).hexdigest())

формат OK: 2452 строк; md5 = a1ff28557631776a9fa7b15727d5a090


## 7. Что можно улучшить

* **Нейросетевой канал.** Bi-encoder (например, `intfloat/multilingual-e5-small`), дообученный на парах
  «запрос → заголовок» из train с in-batch негативами. Он закрыл бы семантические перефразы без общих слов.
  В моём окружении не было доступа к весам моделей, поэтому решение целиком классическое.
* **Сигналы качества объявления** для массовых запросов: CTR и конверсия, если они есть в логах.
* **Больше данных для ранкера.** При переходе с 3 тыс. на 6 тыс. обучающих запросов метрика выросла примерно
  на 0.6 п.п. В train ещё ~11 тыс. подходящих поисков.

**Использованные open-source библиотеки:** pandas, pyarrow, numpy, scipy, scikit-learn (TF-IDF,
SGDClassifier), PyStemmer (Snowball-стеммер), LightGBM. Внешние API и предобученные модели не используются.